# AI Bootcamp — Section 5: How Agents Talk
## Practical 1 · MCP Basics with Ollama (no AWS)

**Goal:** Create a **real local MCP server + client**, discover tools/resources/prompts, execute inventory and shipping tools, and let local Qwen3 explain results.

- All companies, SKU codes, quantities and rates are **fictional training examples**.
- **No AWS / Bedrock / credit card / paid model API** is needed.
- The official course uses **MCP SDK v1**. This notebook deliberately uses the current **MCP Python SDK v2** (`MCPServer`, `Client`) and an **in-memory local transport** so Windows/Jupyter doesn't need extra terminals. This is genuine MCP tool discovery and dispatch, **not** a mock of MCP, but it does **not** demonstrate stdio/network transport or deployment.
- Run code cells **1 → 8 in order** with `Shift+Enter`. If the Python kernel restarts, rerun earlier cells.

**Prerequisites:** Your existing JupyterLab Python 3.12 environment; Ollama Qwen3 `qwen3:1.7b` used in Section 4. Internet may be needed once to install the open-source MCP package.

## Cell 1 — MCP SDK v2 install / upgrade

Cell 1 runs **inside JupyterLab**, not PowerShell. It only installs a Python package, not a paid model. If a package error says `Permission denied` or `SSL`, send the full error; don't paste passwords or API tokens.

In [ ]:
# Cell 1 — Ensure the notebook kernel has the current MCP SDK (v2)
import importlib.metadata as metadata
import subprocess
import sys

try:
    mcp_version = metadata.version("mcp")
except metadata.PackageNotFoundError:
    mcp_version = None

if not mcp_version or mcp_version.split(".")[0] != "2":
    print("Installing/upgrading MCP SDK v2 into the active notebook kernel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mcp>=2,<3"])
    mcp_version = metadata.version("mcp")

assert mcp_version.split(".")[0] == "2", "MCP v2 is required"
print("PASS: MCP SDK v2 ready:", mcp_version)
print("Python:", sys.executable)

## Cell 2 — MCP Server / Client import and Ollama health check

No AI generation happens here. If Ollama isn't running, open Ollama Windows first; the MCP part can still run through Cell 6.

In [ ]:
# Cell 2 — Imports and local Ollama health check
import json
from urllib.request import urlopen, Request
from urllib.error import URLError
from mcp.server import MCPServer
from mcp import Client

OLLAMA_MODEL = "qwen3:1.7b"
OLLAMA_URL = "http://127.0.0.1:11434"

try:
    with urlopen(OLLAMA_URL + "/api/tags", timeout=8) as response:
        tags = json.load(response)
    local_names = [item.get("name", "") for item in tags.get("models", [])]
    ollama_ready = any(name == OLLAMA_MODEL or name.startswith(OLLAMA_MODEL + "-") for name in local_names)
    print("Ollama server: connected")
    print("Qwen3 available:", ollama_ready)
except (URLError, TimeoutError, OSError) as e:
    ollama_ready = False
    print("Ollama not connected yet:", type(e).__name__)

print("PASS: MCP client/server imports available; AWS not required")

## Cell 3 — Create the real MCP Server (tools + resource + prompt)

The MCP Server is named **SwiftCartTrainingMCP**. Two tools return fixed fictional business data; the resource is read-only; the prompt is a reusable template. No real orders are changed.

In [ ]:
# Cell 3 — High-level MCP v2 server, running locally in memory
from typing import Any

INVENTORY = {
    "SKU-101": {"item": "Reusable Bottle", "stock": 12, "price_inr": 499},
    "SKU-102": {"item": "Desk Lamp", "stock": 0, "price_inr": 1299},
}
CITY_BASE_FEE = {"Mumbai": 40, "Ahmedabad": 65, "Delhi": 80}

mcp_server = MCPServer("SwiftCartTrainingMCP")

@mcp_server.tool()
def check_inventory(sku: str) -> dict[str, Any]:
    """Look up training inventory by SKU; this is fictional stock, not live inventory."""
    item = INVENTORY.get(sku.upper())
    if item is None:
        return {"sku": sku, "found": False, "available": False, "stock": 0}
    return {"sku": sku.upper(), "found": True, "available": item["stock"] > 0, **item}

@mcp_server.tool()
def estimate_shipping(city: str, weight_kg: float) -> dict[str, Any]:
    """Estimate fictional shipping cost for Mumbai, Ahmedabad or Delhi."""
    if city not in CITY_BASE_FEE:
        return {"city": city, "valid": False, "reason": "City not supported in this training exercise"}
    if not 0 < weight_kg <= 20:
        return {"city": city, "valid": False, "reason": "Training weight must be > 0 and <= 20 kg"}
    return {"city": city, "valid": True, "weight_kg": weight_kg,
            "shipping_cost_inr": round(CITY_BASE_FEE[city] + 12 * weight_kg, 2)}

@mcp_server.resource("inventory://summary")
def inventory_summary() -> str:
    """Read-only inventory summary for training."""
    return json.dumps(INVENTORY, ensure_ascii=False)

@mcp_server.prompt()
def inventory_advice(customer_question: str) -> str:
    """Hindi instruction template for inventory support questions."""
    return ("इस काल्पनिक inventory data पर आधारित जवाब हिंदी में दें। "
            "कोई अनुमानित stock या real-world price न गढ़ें। सवाल: " + customer_question)

print("PASS: MCP server created — 2 tools, 1 resource, 1 prompt")

## Cell 4 — Test Python business functions locally

This is a pure-Python correctness test before sending calls over MCP.

In [ ]:
# Cell 4 — Rule checks without MCP transport
assert check_inventory("SKU-101")["stock"] == 12
assert check_inventory("SKU-102")["available"] is False
assert check_inventory("MISSING")["found"] is False
assert estimate_shipping("Ahmedabad", 2)["shipping_cost_inr"] == 89.0
assert estimate_shipping("Mumbai", -1)["valid"] is False
print("PASS: local business rules verified")

## Cell 5 — MCP Client discovers server capabilities

`Client(mcp_server)` creates a real protocol client connected **in-process** to the MCP server. The client discovers tools, resources and prompts from the server. JupyterLab supports the top-level `await` below.

In [ ]:
# Cell 5 — Actual MCP client discovery, using v2 in-memory transport
async def discover_mcp():
    async with Client(mcp_server) as client:
        tool_names = [t.name for t in (await client.list_tools()).tools]
        resource_uris = [str(r.uri) for r in (await client.list_resources()).resources]
        prompt_names = [p.name for p in (await client.list_prompts()).prompts]
        assert {"check_inventory", "estimate_shipping"}.issubset(set(tool_names))
        assert "inventory://summary" in resource_uris
        assert "inventory_advice" in prompt_names
        print("MCP tools:", tool_names)
        print("MCP resources:", resource_uris)
        print("MCP prompts:", prompt_names)
        return tool_names, resource_uris, prompt_names

mcp_tools, mcp_resources, mcp_prompts = await discover_mcp()
print("PASS: MCP Client discovered tools, resource and prompt")

## Cell 6 — Real MCP tool calls

The client calls `check_inventory` and `estimate_shipping` **through MCP**, not by calling Python functions directly. Their structured outputs are checked before we trust them.

In [ ]:
# Cell 6 — Call tools over actual MCP and read structured results
async def call_demo_tools():
    async with Client(mcp_server) as client:
        inventory_response = await client.call_tool("check_inventory", {"sku": "SKU-101"})
        shipping_response = await client.call_tool("estimate_shipping", {"city": "Ahmedabad", "weight_kg": 2.0})
        assert not inventory_response.is_error, "Inventory MCP tool reported an error"
        assert not shipping_response.is_error, "Shipping MCP tool reported an error"
        inventory_data = inventory_response.structured_content
        shipping_data = shipping_response.structured_content
        # The SDK can wrap certain structured outputs in a `result` envelope.
        if isinstance(inventory_data, dict) and isinstance(inventory_data.get("result"), dict):
            inventory_data = inventory_data["result"]
        if isinstance(shipping_data, dict) and isinstance(shipping_data.get("result"), dict):
            shipping_data = shipping_data["result"]
        assert isinstance(inventory_data, dict), "MCP inventory output is not structured JSON"
        assert isinstance(shipping_data, dict), "MCP shipping output is not structured JSON"
        assert inventory_data["stock"] == 12
        assert shipping_data["shipping_cost_inr"] == 89.0
        # Also read a resource and render a prompt through MCP
        resource_response = await client.read_resource("inventory://summary")
        prompt_response = await client.get_prompt("inventory_advice", {"customer_question": "SKU-101 उपलब्ध है?"})
        assert resource_response.contents and prompt_response.messages
        return inventory_data, shipping_data

inventory_mcp, shipping_mcp = await call_demo_tools()
print("Inventory tool:", inventory_mcp)
print("Shipping tool:", shipping_mcp)
print("PASS: Real MCP tool calls + resource + prompt completed")

## Cell 7 — Let your local Qwen3 explain the MCP tool results

The **host Python code** chose and called tools in Cell 6. The LLM in Cell 7 only explains the validated results; this is **not autonomous LLM tool selection**. Requests go to `127.0.0.1`, with no AWS or paid API.

In [ ]:
# Cell 7 — Grounded Hindi explanation using local Ollama
payload = {
    "model": OLLAMA_MODEL,
    "stream": False,
    "think": False,
    "messages": [
        {"role": "system", "content": "You explain fictional MCP tool results in concise Hindi. Use ONLY provided facts. Mention this is training data. Do not invent delivery date."},
        {"role": "user", "content": "क्या SKU-101 उपलब्ध है और Ahmedabad में 2 kg shipping कितनी पड़ेगी?\nInventory: "
          + json.dumps(inventory_mcp, ensure_ascii=False) + "\nShipping: " + json.dumps(shipping_mcp, ensure_ascii=False)}
    ],
    "options": {"num_predict": 190, "temperature": 0.1},
}
request = Request(OLLAMA_URL + "/api/chat", data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"})
try:
    with urlopen(request, timeout=240) as response:
        ollama_response = json.load(response)
    mcp_hindi_report = (ollama_response.get("message") or {}).get("content", "").strip()
    assert mcp_hindi_report, "Qwen3 returned an empty answer"
    print(mcp_hindi_report)
    print("\nPASS: Qwen3 explained real MCP tool outputs locally")
except Exception as exc:
    print("Qwen3 call failed:", type(exc).__name__, str(exc)[:250])
    print("MCP tests in Cells 1-6 are still valid. Open Ollama or check qwen3:1.7b, then rerun Cell 7.")
    raise

## Cell 8 — Save your Section 5 MCP Practical 1 evidence

Creates one Markdown report and one JSON evidence file alongside the notebook. Saves only local fictional data and the Qwen3 answer.

In [ ]:
# Cell 8 — Save evidence and narrative
from pathlib import Path
assert mcp_hindi_report.strip(), "Run Cell 7 successfully first"

proof = {
    "section": "5 — How Agents Talk",
    "practical": "1 — Local MCP Basics",
    "sdk": "official MCP Python SDK v2; in-process MCP Client/Server",
    "mcp_tools_discovered": mcp_tools,
    "resources": mcp_resources,
    "prompts": mcp_prompts,
    "inventory_tool_result": inventory_mcp,
    "shipping_tool_result": shipping_mcp,
    "ollama_model": OLLAMA_MODEL,
    "report": mcp_hindi_report,
    "note": "All dataset values are fictional; no real orders or stock were accessed.",
}
Path("section5_mcp_practical1_evidence.json").write_text(json.dumps(proof, ensure_ascii=False, indent=2), encoding="utf-8")
Path("section5_mcp_practical1_report.md").write_text(
    "# Section 5 — MCP Basics (Ollama)\n\n" + mcp_hindi_report
    + "\n\n*Fictional data; local MCP client/server + Qwen3.*\n", encoding="utf-8")
print("PASS: Section 5 MCP Practical 1 completed; two files saved")
print("section5_mcp_practical1_evidence.json")
print("section5_mcp_practical1_report.md")

---
## Next: Section 5 Practical 2 — A2A agent communication

After Cells 1–8 pass and the two files are saved, the next notebook will explore **Agent-to-Agent (A2A)** communication. The original AWS-backed course needs cloud access for its A2A lab; we will replace the model calls with your installed local Ollama and make a separate A2A exercise.

**Troubleshooting:**
- `ModuleNotFoundError: mcp`: rerun Cell 1; if you already imported an old SDK version, restart the kernel and rerun.
- `Client` / `MCPServer` import error: the notebook needs official SDK **v2**; run Cell 1 and restart the kernel.
- Ollama unavailable: verify `ollama list` in **a separate** PowerShell window and that `qwen3:1.7b` exists.
- Do not paste passwords, tokens or AWS keys into notebooks.

**Source syllabus:** [How Agents Talk](https://scaler-content.github.io/How-agents-talk/) · [Bootcamp Section 5 GitHub](https://github.com/pulkit-scaler/weekend-sessions/tree/main/section-05-mcp-and-a2a)  
**SDK reference:** [Official MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk)